# LSTM-Baseline (Count) – Colab-Runner

Trainiert die **reine Zeitreihen-Baseline (LSTM)** und bewertet sie über das
gemeinsame Eval-Modul `shared_eval` (Vergleich 2: Count, MSE/MAE).

Erwartete Struktur in Google Drive (das Repo `bike-link-prediction` hochladen):
```
bike-link-prediction/
├── evaluation/shared_eval.py
├── graphmixer/prepared/   (graphmixer_edges.csv …)
└── lstm/                  (lstm_count.py, dieses Notebook)
```
Zellen von oben nach unten ausführen.

## 1. GPU prüfen

In [ ]:
import torch
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Google Drive einbinden

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Pfade setzen
`ROOT` auf den hochgeladenen Repo-Ordner zeigen lassen (Beispielpfad anpassen).

In [ ]:
import os
ROOT = "/content/drive/MyDrive/bike-link-prediction"
LSTM_DIR = os.path.join(ROOT, "lstm")
EVAL_DIR = os.path.join(ROOT, "evaluation")
PREP_DIR = os.path.join(ROOT, "graphmixer", "prepared")
for p in [LSTM_DIR, EVAL_DIR, PREP_DIR]:
    print(("OK  " if os.path.isdir(p) else "FEHLT ") + p)
fp = os.path.join(PREP_DIR, "graphmixer_edges.csv")
print(("OK  " if os.path.isfile(fp) else "FEHLT ") + fp)

## 4. In den lstm-Ordner wechseln

In [ ]:
%cd "$LSTM_DIR" 

## 5. Konfiguration wählen
Für einen schnellen Test `epochs=2`; für den vollen Lauf `epochs=10` (Default).

In [ ]:
from lstm_count import LSTMConfig
cfg = LSTMConfig(epochs=10)   # z. B. LSTMConfig(epochs=2) für Smoke-Test
print("Lookback:", cfg.lookback, "| hidden:", cfg.hidden_dim,
      "| epochs:", cfg.epochs, "| max_train_samples:", cfg.max_train_samples)

## 6. Training + Bewertung starten
Trainiert das globale LSTM, exportiert Vorhersagen und gibt MSE/MAE/RMSE aus.
Ergebnis: `lstm/predictions/lstm_pred_{val,test}.csv`.

In [ ]:
from lstm_count import main
main(cfg)

## 7. Ergebnisse erneut bewerten
Dieselbe Bewertung, die später auch der Count-Kopf des Hybridmodells nutzt.

In [ ]:
import sys, pandas as pd
sys.path.insert(0, EVAL_DIR)
from shared_eval import SharedLinkEval
ev = SharedLinkEval()
for split in ["val", "test"]:
    pred = pd.read_csv(f"predictions/lstm_pred_{split}.csv")
    res = ev.score_count(pred, split=split)
    print(f"[{split}] MSE={res['mse']:.4f} MAE={res['mae']:.4f} RMSE={res['rmse']:.4f} (n={res['n_total']})")
pred.head()

## Nächste Schritte
- MSE/MAE notieren – das sind die Leitmetriken für Vergleich 2.
- Dieselben `predictions/*.csv` dienen später dem direkten Vergleich gegen den
  **Count-Kopf des Hybridmodells** (`ev.score_count`).